# Orca Nano — QLoRA Fine-Tune (Colab free T4)

Run cells top to bottom. Before starting: **Runtime → Change runtime type → T4 GPU**.

This is a smaller-VRAM config than Orca's cloud A100 preset — rank 16 LoRA,
2048 max sequence length, 4-bit quant — sized to actually fit a free T4's
16GB, not just copy the A100 settings and OOM.

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" trl transformers datasets peft bitsandbytes accelerate

## Upload your training data

Upload the two files from your machine: `orca_llama3_train.jsonl` and `orca_llama3_eval.jsonl`
(found at `~/.orca/training/formatted/` on your local machine).

In [ ]:
from google.colab import files
uploaded = files.upload()  # select orca_llama3_train.jsonl and orca_llama3_eval.jsonl
print('Uploaded:', list(uploaded.keys()))

In [ ]:
import json

def load_jsonl(path):
    lines = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    lines.append(json.loads(line))
                except Exception:
                    pass
    return lines

raw_train = load_jsonl('orca_llama3_train.jsonl')
raw_eval  = load_jsonl('orca_llama3_eval.jsonl') if 'orca_llama3_eval.jsonl' in uploaded else raw_train[:max(1, len(raw_train)//10)]
print(f'train={len(raw_train)} eval={len(raw_eval)}')

## Load base model (4-bit) + attach LoRA

Rank 16 (not 128 like the A100 preset) — T4 has 16GB VRAM, a bigger rank
risks an out-of-memory crash partway through training.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # smaller than the 4096 cloud preset — fits T4 memory
base_model = "unsloth/Qwen2.5-7B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
from datasets import Dataset

def format_conv(ex):
    turns = ex.get("conversations", ex.get("text"))
    if isinstance(turns, str):
        return turns  # already-formatted llama3 text field
    parts = []
    for t in turns:
        role = t.get("role", "")
        val  = t.get("value", "")
        if role == "system":
            parts.append(f"<|start_header_id|>system<|end_header_id|>\n\n{val}<|eot_id|>")
        elif role == "human":
            parts.append(f"<|start_header_id|>user<|end_header_id|>\n\n{val}<|eot_id|>")
        elif role == "gpt":
            parts.append(f"<|start_header_id|>assistant<|end_header_id|>\n\n{val}<|eot_id|>")
    return "".join(parts)

# The formatter.py output already has a 'text' field per example (llama3 format) — use it directly if present.
train_ds = Dataset.from_list([{"text": ex["text"] if "text" in ex else format_conv(ex)} for ex in raw_train])
eval_ds  = Dataset.from_list([{"text": ex["text"] if "text" in ex else format_conv(ex)} for ex in raw_eval])
print(f'train_ds={len(train_ds)} eval_ds={len(eval_ds)}')

## Train

Batch size 2 + grad accumulation 4 (effective batch 8) — smaller than the
cloud preset's batch 8, again sized for T4's 16GB rather than an A100's 40GB.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import time

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        max_grad_norm=1.0,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_steps=100,
        save_strategy="no",
        output_dir="output",
        eval_strategy="steps",
        report_to="none",
    ),
)

print("[train] starting QLoRA training...")
t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 60
print(f"[train] done in {elapsed:.1f} min")

## Merge LoRA + export GGUF, then download

In [ ]:
print("[merge] merging LoRA adapters...")
model.save_pretrained_merged("merged", tokenizer, save_method="merged_16bit")
print("[merge] saved to ./merged")

print("[gguf] converting to GGUF q4_k_m...")
model.save_pretrained_gguf("gguf", tokenizer, quantization_method="q4_k_m")
print("[gguf] saved to ./gguf")

In [ ]:
import glob
from google.colab import files

gguf_files = glob.glob("gguf/*q4_k_m*.gguf")
print("Found:", gguf_files)
if gguf_files:
    files.download(gguf_files[0])  # downloads to your machine's Downloads folder
else:
    print("No GGUF file found — check the [gguf] step above for errors.")